In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [29]:
print('---LOAD CLEANED DATA ---')

file_path = "data/processed/antimicrobial_clean.csv"
print("Loading dataset...\n")
df = pd.read_csv(file_path)

print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}\n")
df.info()
print()
display(df.head())

In [30]:
# Drop rows where Inhibition Zone is null
df_model = df.dropna(subset=['Inhibition Zone (mm)']).copy()
print(f"Rows after dropping null IZ: {df_model.shape[0]}")

In [31]:
print('---Define Target Variable(y) And Features(X) ---')
#Target Variable
y = df_model['Inhibition Zone (mm)']
X = df_model.drop(columns=['DOI','Inhibition Zone (mm)','YEAR'] ,errors='ignore')
print(f"\nFeature columns (X): {list(X.columns)}")
print(f"Target variable (y): Inhibition Zone (mm), shape: {y.shape}")



In [32]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/processed/antimicrobial_clean.csv")

df_model = df.dropna(subset=['Inhibition Zone (mm)']).copy()

doi_counts = df_model['DOI'].groupby(df_model['DOI']).transform('count')
df_small = df_model[doi_counts <= 4]
df_large = df_model[doi_counts > 4]

groups = pd.factorize(df_small['DOI'])[0]
gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

train_pos, test_pos = next(gss.split(df_small, groups=groups))

test_indices = df_small.iloc[test_pos].index
train_indices = df_small.iloc[train_pos].index.union(df_large.index)

train_df = df_model.loc[train_indices]
test_df = df_model.loc[test_indices]

In [33]:
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X_train = train_df.drop(columns=['DOI', 'Inhibition Zone (mm)', 'YEAR'], errors='ignore')
y_train = train_df['Inhibition Zone (mm)']

X_test = test_df.drop(columns=['DOI', 'Inhibition Zone (mm)', 'YEAR'], errors='ignore')
y_test = test_df['Inhibition Zone (mm)']

categorical_cols = [
    'MATERIAL',
    'PRECURSOR',
    'Synthesis_Method',
    'Plant Extract Type',
    'Bacterial Strain',
    'ROS Generation',
    'Gram Status',
    'Resistant Type',
    'IZ_Censored'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

os.makedirs("data/splits", exist_ok=True)

pd.DataFrame(X_train_encoded).to_csv("data/splits/X_train.csv", index=False)
pd.DataFrame(X_test_encoded).to_csv("data/splits/X_test.csv", index=False)
y_train.to_csv("data/splits/y_train.csv", index=False)
y_test.to_csv("data/splits/y_test.csv", index=False)


In [34]:
!git clone https://github.com/shaivis08/antimicrobial-ml.git


In [37]:

!git add -f data/splits/*.csv

!git commit --amend -m "data: add clean encoded train and test splits to data/splits/"

!git push --force origin main

In [39]:
# 1. Add all modified files (both the notebook and the split CSVs)
!git add -u
!git add -f data/splits/*.csv

# 2. Commit the changes
!git commit -m "refactor: complete categorical encoding and export clean splits"

# 3. Push to GitHub using your PAT token


In [40]:
!git pull --rebase origin main


In [41]:
!git rebase --abort

In [42]:

!git add -f data/splits/*.csv
!git add -u
!git commit -m "data: add fully one-hot encoded train and test splits"

In [ ]:
import google.colab
import json

nb_json = google.colab._message.blocking_request('get_ipynb')
with open("notebooks/02_feature_engineering.ipynb", "w", encoding="utf-8") as f:
    json.dump(nb_json['ipynb'], f, indent=2)

print(" Notebook re-saved cleanly without token exposures.")